# 09 - Agentic RAG

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval

## What this notebook covers

- What makes RAG agentic vs standard RAG
- The ReAct pattern - Reason + Act in a retrieval loop
- Multi-hop retrieval - following chains of references
- Self-reflection - FLARE and adaptive retrieval
- Memory in agentic RAG - short-term scratchpad vs long-term store
- Tool use vs RAG - when each is the right architecture
- Connection to real agentic systems built in this course

**Pure Python only - no external dependencies.**

---
## 1. Standard RAG vs Agentic RAG

**Standard RAG** (what local-rag-llm implements):
```
Query -> Retrieve once -> Generate answer
```
Fixed pipeline. One retrieval step. Deterministic order.
The LLM has no control over what gets retrieved or when.

**Agentic RAG:**
```
Query -> LLM decides: do I need to retrieve? what? -> Retrieve
      -> LLM reads result: is this enough? -> Retrieve again?
      -> LLM synthesises answer from multi-hop context
```
The LLM is an active participant in the retrieval process.
It can retrieve multiple times, from multiple sources, in any order.
It decides when it has enough information to answer.

**When standard RAG is sufficient:**
- Single document, single question, single retrieval covers it
- Query is well-specified and self-contained
- Answer exists in one contiguous section of the document

**When you need agentic RAG:**
- Answer requires information from multiple disconnected sections
- Question requires comparison across documents
- User question requires first answering a sub-question
- The answer depends on the retrieved content (unpredictable retrieval path)

In [ ]:
# Standard vs Agentic RAG - pipeline comparison

standard_rag_steps = [
    "1. User query received",
    "2. Query rewritten (if follow-up)",
    "3. FAISS retrieval (k=16)",
    "4. BM25 retrieval (k=4)",
    "5. Merge + junk filter + CrossEncoder rerank",
    "6. Top 5 chunks assembled as context",
    "7. Claude generates answer",
    "8. Output validation",
    "9. Answer returned",
]

agentic_rag_steps = [
    "1. User query received",
    "2. LLM reasons: what do I need to answer this?",
    "3. LLM calls retrieve_tool(query='first sub-question')",
    "4. LLM reads result: is this complete? do I need more?",
    "5. LLM calls retrieve_tool(query='follow-up based on result')",
    "6. LLM reads result: compares with previous retrieval",
    "7. [Optional] LLM calls another tool - calculator, API, DB",
    "8. LLM synthesises final answer from all retrieved context",
    "9. Answer returned",
]

print("STANDARD RAG (local-rag-llm)")
print("=" * 50)
for step in standard_rag_steps:
    print(f"  {step}")

print()
print("AGENTIC RAG")
print("=" * 50)
for step in agentic_rag_steps:
    print(f"  {step}")

print()
print("Key difference: in agentic RAG, steps 3-6 can repeat N times.")
print("The LLM decides when to stop retrieving. Standard RAG retrieves exactly once.")

---
## 2. The ReAct Pattern

ReAct (Reason + Act) is the foundational pattern for agentic RAG.
It interleaves reasoning steps with action steps in a loop.

```
Thought: I need to find X before I can answer Y
Action: retrieve("X")
Observation: [retrieved content about X]
Thought: Now I know X. But to answer Y I also need Z.
Action: retrieve("Z")
Observation: [retrieved content about Z]
Thought: Now I have enough. I can answer Y.
Action: answer("Based on X and Z, Y is...")
```

Each Thought-Action-Observation cycle is one hop.
The LLM decides how many hops are needed.

ReAct was introduced in the Yao et al. 2022 paper.
It is the basis of most modern agent frameworks:
LangChain AgentExecutor, LlamaIndex ReAct Agent, AutoGen, CrewAI.

In [ ]:
# ReAct pattern - simulating a multi-hop retrieval agent

# Knowledge base: document chunks indexed by topic
knowledge_base = {
    "ZAA definition": "Zero Ambient Authority means agents must never inherit the developer's full administrative privileges.",
    "JIT downscoping": "JIT downscoping provides hyper-restricted credentials scoped to the exact data sources required for the task.",
    "SPIFFE IDs": "SPIFFE IDs are cryptographic identifiers assigned to every individual agent in the system.",
    "Confused Deputy": "The Confused Deputy problem occurs when a prompt injection tricks an over-privileged agent into executing unauthorised commands.",
    "Confused Deputy fix": "To resolve Confused Deputy, agents must authenticate using a dedicated agentic identity, not delegated user credentials.",
    "ABAC": "Attribute-Based Access Control evaluates access based on subject, resource, action, and environment attributes.",
]

def retrieve(query):
    query_lower = query.lower()
    results = []
    for topic, content in knowledge_base.items():
        topic_words = set(topic.lower().split())
        query_words = set(query_lower.split())
        if len(topic_words & query_words) > 0:
            results.append(f"[{topic}]: {content}")
    return results if results else ["No relevant content found."]

# Simulating a ReAct agent answering a multi-hop question
question = "What problem does agentic identity solve, and what is the fix?"

print(f"Question: '{question}'")
print("=" * 65)
print()

# Hop 1: Agent reasons it needs to find the problem first
thought_1 = "I need to find what problem agentic identity is related to. The Confused Deputy problem is mentioned in IAM context."
action_1 = "retrieve('Confused Deputy')"
result_1 = retrieve("Confused Deputy")

print(f"Thought 1: {thought_1}")
print(f"Action 1:  {action_1}")
print(f"Observation 1:")
for r in result_1:
    print(f"  {r}")
print()

# Hop 2: Agent now knows the problem, retrieves the fix
thought_2 = "I found the Confused Deputy problem. Now I need the specific fix - agentic identity."
action_2 = "retrieve('Confused Deputy fix')"
result_2 = retrieve("Confused Deputy fix")

print(f"Thought 2: {thought_2}")
print(f"Action 2:  {action_2}")
print(f"Observation 2:")
for r in result_2:
    print(f"  {r}")
print()

# Hop 3: Agent has both pieces, generates answer
thought_3 = "I now have both the problem and the fix. I can synthesise the answer."
answer = ("The Confused Deputy problem is what agentic identity solves. " +
           "It occurs when prompt injection tricks an over-privileged agent into executing " +
           "unauthorised commands. The fix: agents must use a dedicated agentic identity " +
           "rather than delegated user credentials, limiting their permissions to only what each task requires.")

print(f"Thought 3: {thought_3}")
print(f"Final answer: {answer}")
print()
print(f"Hops required: 2 retrieval steps")
print(f"Standard RAG: single retrieval might only find the problem OR the fix, not both.")

---
## 3. Multi-Hop Retrieval

Some questions cannot be answered from a single retrieval.
They require following a chain of references across documents or sections.

**Example multi-hop chain:**
```
Q: "What mechanism prevents the Confused Deputy problem from affecting production databases?"

Hop 1: retrieve("Confused Deputy") -> learns: caused by over-privileged agents
Hop 2: retrieve("agentic identity") -> learns: SPIFFE IDs, separate agentic credentials  
Hop 3: retrieve("production database access control") -> learns: JIT tokens scope to specific DB
Synthesis: The combination of agentic identity + JIT downscoping prevents it
```

Multi-hop is critical for D365 enterprise RAG:
- User asks about a process that spans multiple D365 modules
- Answer requires GL config + vendor setup + approval workflow - three separate topics
- Each retrieval depends on what the previous one returned

In [ ]:
# Multi-hop retrieval - bridging entity example

# Simulating a knowledge graph retrieval where answers chain
entity_knowledge = {
    "ephemeral sandbox": {
        "definition": "A network-isolated container that resets state completely between runs.",
        "purpose": "Contains blast radius of vibe-coded scripts.",
        "see_also": ["gVisor", "container escape"],
    },
    "gVisor": {
        "definition": "A kernel-level sandbox from Google that intercepts system calls.",
        "how": "Runs an application kernel in userspace, intercepting all syscalls before they reach the host.",
        "see_also": ["ephemeral sandbox", "container isolation"],
    },
    "container escape": {
        "definition": "An attack where malicious code breaks out of container isolation to access the host.",
        "relevance": "Vibe-coded scripts may attempt this if they contain vulnerabilities.",
        "mitigation": "gVisor kernel isolation prevents container escape from reaching the host kernel.",
        "see_also": ["gVisor", "ephemeral sandbox"],
    },
}

def hop_retrieve(topic):
    return entity_knowledge.get(topic.lower(), {})

question = "How does gVisor prevent vibe-coded scripts from compromising the host?"

print(f"Question: '{question}'")
print("=" * 65)
print()

# Hop 1: Start with gVisor
hop1 = hop_retrieve("gVisor")
print("Hop 1: retrieve('gVisor')")
print(f"  Definition: {hop1.get('definition', 'not found')}")
print(f"  How: {hop1.get('how', 'not found')}")
print(f"  See also: {hop1.get('see_also', [])}")
print()

# Hop 2: Follow the 'container escape' reference from hop 1
hop2 = hop_retrieve("container escape")
print("Hop 2: retrieve('container escape') [followed from see_also]")
print(f"  Definition: {hop2.get('definition', 'not found')}")
print(f"  Relevance: {hop2.get('relevance', 'not found')}")
print(f"  Mitigation: {hop2.get('mitigation', 'not found')}")
print()

print("Multi-hop synthesis:")
print(f"  gVisor intercepts system calls in userspace (Hop 1).")
print(f"  Container escape attempts hit gVisor's userspace kernel, not the host (Hop 2).")
print(f"  A vibe-coded script with vulnerabilities cannot reach the actual host kernel.")
print()
print("Standard RAG: single retrieve('gVisor') returns the definition but not the container escape link.")
print("Multi-hop: follows the reference chain to build a complete mechanistic answer.")

---
## 4. Self-Reflection and FLARE

**The problem with standard RAG:**
The LLM generates an answer from retrieved context even if the context is incomplete.
It cannot say "I don't have enough information yet - retrieve more."

**Self-reflection:** After generating a response, the LLM evaluates its own answer:
- Is this grounded in the retrieved content?
- Are there claims I made that I'm not confident about?
- Should I retrieve more before finalising?

**FLARE (Forward-Looking Active REtrieval):**
A specific self-reflection technique:
1. LLM generates tentative answer tokens one by one
2. When it generates a low-confidence token, it stops
3. Retrieves documents related to the uncertain claim
4. Regenerates that section with the additional context

FLARE is complex to implement. A simpler version: **post-generation reflection.**
After generating, ask the LLM: 'Are there claims in this answer not supported by the retrieved context?
If so, what would you need to retrieve to verify them?'

In [ ]:
# Self-reflection pattern - post-generation verification

retrieved_context = [
    "The Green Team executes a Stateful Quarantine via SOAR playbooks.",
    "This gracefully revokes the agent's tool access while preserving memory for forensic analysis.",
    "The Green Team also performs Auto-Refactoring to autonomously patch vulnerabilities.",
]

generated_answer = (
    "The Green Team responds to security incidents by quarantining compromised agents. "
    "It uses SOAR playbooks to revoke tool access while preserving memory. "
    "Auto-Refactoring rewrites vulnerable code autonomously. "
    "The Green Team also coordinates with external SIEM systems for enterprise reporting. "  # hallucinated
)

def check_grounding(answer, context):
    sentences = [s.strip() for s in answer.replace('. ', '.|').split('|') if s.strip()]
    context_words = set(' '.join(context).lower().split())
    results = []
    for sent in sentences:
        sent_words = set(sent.lower().split())
        overlap = len(sent_words & context_words) / len(sent_words) if sent_words else 0
        grounded = overlap > 0.3
        results.append((sent, overlap, grounded))
    return results

print("SELF-REFLECTION - grounding check")
print("=" * 65)
print("Retrieved context:")
for c in retrieved_context:
    print(f"  - {c}")
print()
print("Generated answer sentences:")
grounding = check_grounding(generated_answer, retrieved_context)
print(f"{"Grounded":<10} {"Overlap":<9} Sentence")
print("-" * 70)
for sent, overlap, grounded in grounding:
    status = "YES" if grounded else "NO - REVIEW"
    flag = " << POSSIBLE HALLUCINATION" if not grounded else ""
    print(f"{status:<10} {overlap:<9.2f} '{sent[:55]}'{flag}")

print()
print("Low-grounded sentence flagged for retrieval:")
low = [s for s, o, g in grounding if not g]
if low:
    print(f"  '{low[0]}'")
    print(f"  Action: retrieve('Green Team SIEM enterprise reporting') to verify or discard this claim.")

---
## 5. Memory in Agentic RAG

Agentic RAG systems need memory to function across turns and tasks.
Three types of memory serve different purposes.

**Scratchpad (in-context, ephemeral)**
The agent's working memory for a single task.
All thoughts, retrieved chunks, and intermediate results live here.
Disappears when the context window ends.
This is what the ReAct Thought-Action-Observation loop uses.

**Short-term conversation memory (session-scoped)**
The conversation history for the current session.
local-rag-llm implements this with sliding window + summarisation.
Lets the LLM reference earlier questions and answers.

**Long-term memory (persistent, cross-session)**
Facts that should persist across multiple conversations.
Typically stored in a vector database as embeddings.
Example: user preferences, domain-specific learned facts, previous query patterns.
Not yet in local-rag-llm - on the roadmap.

**Memory hierarchy in production agentic systems:**
```
L1: Scratchpad (context window)     - current task only, microsecond access
L2: Session history (DB)            - current session, millisecond access  
L3: Vector memory (FAISS/Pinecone)  - cross-session facts, ~100ms access
L4: Document store (RAG index)      - full knowledge base, ~500ms access
```

In [ ]:
# Memory types illustrated

class AgentMemory:
    def __init__(self):
        self.scratchpad = []        # current task working memory
        self.session_history = []   # conversation turns this session
        self.long_term = {}         # persistent facts across sessions

    def add_thought(self, thought):
        self.scratchpad.append({"type": "thought", "content": thought})

    def add_observation(self, observation):
        self.scratchpad.append({"type": "observation", "content": observation})

    def commit_to_session(self, user_msg, assistant_msg):
        self.session_history.append({"user": user_msg, "assistant": assistant_msg})
        self.scratchpad = []  # clear scratchpad after task

    def store_long_term(self, key, value):
        self.long_term[key] = value

    def recall_long_term(self, query):
        query_words = set(query.lower().split())
        results = []
        for key, value in self.long_term.items():
            key_words = set(key.lower().split())
            if key_words & query_words:
                results.append((key, value))
        return results

    def show_state(self):
        print(f"Scratchpad ({len(self.scratchpad)} items):")
        for item in self.scratchpad:
            print(f"  [{item['type']}] {item['content'][:70]}")
        print(f"Session history ({len(self.session_history)} turns):")
        for turn in self.session_history[-2:]:
            print(f"  User: {turn['user'][:50]}")
            print(f"  Asst: {turn['assistant'][:50]}")
        print(f"Long-term memory ({len(self.long_term)} items):")
        for key, val in self.long_term.items():
            print(f"  [{key}] {str(val)[:60]}")

# Simulate an agent working through a multi-turn interaction
memory = AgentMemory()

# Task 1: Answer a question
memory.add_thought("I need to find what JIT downscoping is")
memory.add_observation("JIT downscoping: hyper-restricted credentials scoped to exact task")
memory.add_thought("I have enough context to answer")

memory.commit_to_session("What is JIT downscoping?", "JIT downscoping provides task-scoped credentials that expire immediately.")

# Store useful fact for future sessions
memory.store_long_term("user asks about IAM frequently", "User has D365 IAM background, prefers technical detail")

# Task 2: New question this session - can reference history
memory.add_thought("User is asking a follow-up. From history I know we discussed JIT downscoping.")
memory.add_observation("SPIFFE IDs are assigned per agent, combined with JIT for full IAM picture")

print("AGENT MEMORY STATE - after two tasks")
print("=" * 55)
memory.show_state()
print()
print("Long-term memory survives session end.")
print("Scratchpad was cleared after Task 1 - new scratchpad for Task 2.")
print("Session history available for context in same session.")

---
## 6. Tool Use vs RAG - Architecture Decision

Agentic systems can retrieve information two ways.
Choosing the wrong one for the use case is a common mistake.

**RAG (Retrieval Augmented Generation)**
Best for: unstructured text - documents, manuals, reports, whitepapers
Mechanism: embed query, find similar text chunks, pass to LLM
When: the answer is *expressed in natural language* in a document

**Tool use (API / function calls)**
Best for: structured data - databases, APIs, live systems
Mechanism: LLM calls a function, gets back structured data
When: the answer is in a database record, a live API, or requires computation

**D365 RAG decision matrix:**

| Question type | Right approach |
|---|---|
| What does the ZAA policy say? | RAG (policy document) |
| What is vendor CONTOSO-001 credit limit? | Tool use (VendTable API call) |
| How do I configure approval workflows? | RAG (setup documentation) |
| What invoices are overdue today? | Tool use (live DB query) |
| What are the best practices for GL reconciliation? | RAG (procedural document) |
| What is the current USD-SGD exchange rate? | Tool use (FX API) |

**Hybrid architecture (what real D365 AI assistants need):**
The LLM decides at runtime whether to call RAG or a D365 API tool.
The Presales Demo Provisioning Agent capstone uses this pattern:
governing agent decides which sub-agent handles each task.

In [ ]:
# Tool use vs RAG decision illustration

def rag_retrieve(query):
    """Simulates RAG retrieval - finds relevant document chunks"""
    doc_knowledge = {
        "zaa policy": "Zero Ambient Authority policy requires agents use JIT-scoped credentials.",
        "approval workflow config": "Configure approval workflows in PurchApprovalJournal settings.",
        "gl reconciliation": "GL reconciliation best practice: match subledger to GL daily.",
    }
    for key, content in doc_knowledge.items():
        if any(word in query.lower() for word in key.split()):
            return f"[RAG] {content}"
    return "[RAG] No relevant documentation found."

def tool_call_d365(function_name, params):
    """Simulates a D365 OData API call"""
    mock_db = {
        ("VendTable", "CONTOSO-001"): {"CreditMax": 50000, "PaymTermId": "NET30", "Blocked": False},
        ("CustInvoiceTable", "overdue"): [{"InvoiceId": "INV-2026-0421", "Amount": 12500, "DaysOverdue": 15}],
        ("ExchangeRate", "USD-SGD"): {"Rate": 1.342, "Date": "2026-07-09"},
    }
    result = mock_db.get((function_name, params))
    return f"[TOOL:{function_name}({params})] {result}" if result else "[TOOL] No data found."

# Agent routing decision
queries = [
    ("What does the ZAA policy say?",                   "rag",  "zaa policy",        None),
    ("What is vendor CONTOSO-001 credit limit?",         "tool", "VendTable",         "CONTOSO-001"),
    ("How do I configure approval workflows?",           "rag",  "approval workflow",  None),
    ("What invoices are overdue today?",                 "tool", "CustInvoiceTable",   "overdue"),
    ("What is the current USD-SGD exchange rate?",       "tool", "ExchangeRate",       "USD-SGD"),
    ("What are GL reconciliation best practices?",       "rag",  "gl reconciliation",  None),
]

print("TOOL USE vs RAG - D365 Agent Routing")
print("=" * 65)
print(f"{"Query":<45} {"Route":>6}  Result")
print("-" * 80)
for query, route, param1, param2 in queries:
    if route == "rag":
        result = rag_retrieve(param1)
    else:
        result = tool_call_d365(param1, param2)
    print(f"{query:<45} [{route.upper()}]")
    print(f"  {result[:70]}")
    print()

---
## 7. Connection to Built Projects

The agentic RAG concepts in this notebook appear directly in two projects built this course:

**local-rag-llm (standard RAG with agentic elements):**
- Query rewriting = single-hop reasoning about the query before retrieval
- Conversation summarisation = short-term session memory management
- 3-layer security = guardrails on agent actions (Pillar 4 in the Vibe Coding whitepaper)
- Next step: add parent-document + multi-query = approaches true agentic retrieval

**Presales Demo Provisioning Agent (capstone - full agentic system):**
- Governing agent = orchestrator that decides which sub-agent handles each task
- Governance gates = the agent's self-reflection step (can I proceed with this action?)
- Cost estimator = tool call to Azure pricing API before provisioning
- Terraform provisioner = tool execution, not RAG - correct routing decision
- Tender P&L backend = another tool call, not retrieval

The capstone is the natural evolution of what local-rag-llm started:
from 'LLM reads documents' to 'LLM orchestrates tools and retrieval together'.

In [ ]:
# Capstone architecture mapped to agentic RAG concepts

capstone_components = [
    {
        "component": "Governing Agent",
        "agentic_rag_concept": "Orchestrator / ReAct loop",
        "what_it_does": "Receives request, decides which sub-agent handles each step",
        "memory_type": "Scratchpad (task state) + session history",
    },
    {
        "component": "Governance Gate",
        "agentic_rag_concept": "Self-reflection / circuit breaker",
        "what_it_does": "Agent checks: is this request authorised? Does it meet policy?",
        "memory_type": "Policy rules (long-term, read-only)",
    },
    {
        "component": "Cost Estimator",
        "agentic_rag_concept": "Tool use (not RAG)",
        "what_it_does": "Calls Azure Retail Pricing API - live structured data needed",
        "memory_type": "None - stateless API call",
    },
    {
        "component": "Terraform Provisioner",
        "agentic_rag_concept": "Tool execution with state",
        "what_it_does": "Executes infrastructure changes - irreversible action",
        "memory_type": "Deployment state store (long-term)",
    },
    {
        "component": "Tender P&L Backend",
        "agentic_rag_concept": "Tool use (database query)",
        "what_it_does": "Queries financial data - structured, not document retrieval",
        "memory_type": "Financial DB (external)",
    },
]

print("CAPSTONE vs AGENTIC RAG CONCEPTS")
print("=" * 70)
for comp in capstone_components:
    print(f"\n{comp["component"]}")
    print(f"  Concept:  {comp["agentic_rag_concept"]}")
    print(f"  Does:     {comp["what_it_does"]}")
    print(f"  Memory:   {comp["memory_type"]}")

print()
print("Key insight: the capstone correctly uses TOOL CALLS for structured data")
print("(Azure pricing, Terraform, financial DB) and would use RAG for")
print("unstructured data (tender specifications, compliance documentation).")
print("Mixing these up - using RAG for DB queries or tool calls for documents -")
print("is the most common architectural mistake in enterprise AI systems.")

---
## 8. Summary

| Concept | Key point |
|---|---|
| **Standard RAG** | Fixed pipeline: retrieve once, generate once |
| **Agentic RAG** | LLM controls retrieval: decides when, what, how many hops |
| **ReAct** | Interleave Thought-Action-Observation cycles - foundation of all agent frameworks |
| **Multi-hop** | Follow chains of references across documents to build complete answers |
| **Self-reflection** | LLM evaluates its own answer for grounding before returning it |
| **FLARE** | Stop generation at uncertain tokens, retrieve more, continue - complex but powerful |
| **Scratchpad** | In-context working memory for current task - ephemeral |
| **Session memory** | Conversation history in current session - local-rag-llm uses sliding window |
| **Long-term memory** | Cross-session persistent facts - vector DB or structured store |
| **Tool vs RAG** | RAG for unstructured text; tool calls for structured DB/API data |
| **D365 hybrid** | Policy docs -> RAG; vendor records, invoices, FX rates -> tool calls |

**The agentic RAG insight:**
Standard RAG is a pipeline. Agentic RAG is a loop.
The difference is whether the LLM participates in deciding what to retrieve
or merely consumes whatever the fixed pipeline gives it.
For simple document Q&A, the pipeline is more reliable and cheaper.
For enterprise workflows spanning multiple systems, the loop is essential.

---
## Series Complete

This series covered the full RAG stack:

- **00** - RAG fundamentals (what, why, the pipeline)
- **01** - Chunking strategies (RecursiveCharacterTextSplitter, overlap, semantic, parent-document)
- **02** - Embeddings and vector stores (FAISS internals, distance metrics, nomic-embed-text)
- **03** - Sparse retrieval and BM25 (TF-IDF, the formula, D365 critical use cases)
- **04** - Hybrid retrieval (RRF, weighted fusion, junk filter, tuning guide)
- **05** - Reranking with CrossEncoders (bi-encoder vs cross-encoder, ms-marco, cost analysis)
- **06** - RAG evaluation with RAGAS (metrics, golden dataset discipline, NaN bug)
- **07** - RAG security (prompt injection, 3-layer defence, fail-open vs fail-closed)
- **08** - Advanced RAG patterns (parent-document, multi-query, HyDE, self-querying)
- **09** - Agentic RAG (ReAct, multi-hop, self-reflection, memory, tool vs RAG)